# Parallel 3 IDK Cascades

Clean real-time CPU/MPS multiprocessing notebook. This version has no simulation test cells and no cached-logit test cells.

## Imports

This cell imports the libraries used by the notebook. The worker import happens in the constants cell because it needs the project path first.

In [1]:
import json
import sys
import tarfile
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.multiprocessing as mp
from PIL import Image
from torch.utils.data import DataLoader, IterableDataset
from torchvision import transforms
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.svm import SVC


## Constants

Edit this cell to change models, devices, paths, worker counts, thresholds, sample counts, and output files. Model A runs on CPU; Model B and Model C run on MPS.

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RUNS_DIR = PROJECT_ROOT / "runs"
IMAGENETV2_DIR = PROJECT_ROOT / "ImageNet-V2 DataSet"
RUNS_DIR.mkdir(exist_ok=True)

VARIANT_ARCHIVES = {
    "matched-frequency": IMAGENETV2_DIR / "imagenetv2-matched-frequency.tar.gz",
    "threshold-0.7": IMAGENETV2_DIR / "imagenetv2-threshold0.7.tar.gz",
    "top-images": IMAGENETV2_DIR / "imagenetv2-top-images.tar.gz",
}

MODEL_A = "resnet18"
MODEL_B = "resnet34"
MODEL_C = "resnet50"

MODEL_A_DEVICE = "mps"
MODEL_B_DEVICE = "mps"
MODEL_C_DEVICE = "mps"

MODELS = (MODEL_A, MODEL_B, MODEL_C)
MPS_MODELS = (MODEL_B, MODEL_C)
MODEL_DEVICES = {
    MODEL_A: MODEL_A_DEVICE,
    MODEL_B: MODEL_B_DEVICE,
    MODEL_C: MODEL_C_DEVICE,
}

WORKER_LIMITS = {"cpu": 1, "mps": 3}
MAX_SAMPLES = 10000
BATCH_SIZE = 1
CONFIDENCE_THRESHOLD = 0.7
MAX_IN_FLIGHT_PER_MPS_MODEL = 1000

ROUTER_TRAIN_PREFIXES = ("matched", "top")
TEST_VARIANT = "threshold-0.7"
ROUTER_REQUIRES_CORRECT = False
SAVE_RESULTS = True
RESULTS_PATH = RUNS_DIR / "real_cpu_mps_worker_results.json"
PREDICTIONS_PATH = RUNS_DIR / "real_cpu_mps_worker_predictions.npz"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from real_time_mp_workers import model_worker


## Image Transform

This cell defines the ImageNet preprocessing used for router training and for the real-time test images.

In [3]:
preprocess = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


## Dataset Loader

This cell reads ImageNetV2 rows from the local tar archives, applies the ImageNet transform, and returns a DataLoader for the real-time test.

In [4]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")


def label_from_key(key):
    return int(key.split("/")[1])


def stream_imagenet_v2_rows(variant, max_samples):
    emitted = 0
    with tarfile.open(VARIANT_ARCHIVES[variant], "r:*") as tar:
        for member in tar:
            if not member.isfile() or not member.name.lower().endswith(IMAGE_EXTENSIONS):
                continue

            image_file = tar.extractfile(member)
            if image_file is None:
                continue
            image = Image.open(image_file).convert("RGB")
            image_file.close()

            yield image, label_from_key(member.name)
            emitted += 1
            if emitted >= max_samples:
                break


class StreamingImageNetV2Dataset(IterableDataset):
    def __init__(self, variant, max_samples):
        self.variant = variant
        self.max_samples = int(max_samples)

    def __iter__(self):
        for image, label in stream_imagenet_v2_rows(self.variant, self.max_samples):
            yield preprocess(image), int(label)

    def __len__(self):
        return self.max_samples


def make_streaming_loader(variant, max_samples):
    dataset = StreamingImageNetV2Dataset(variant, max_samples)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        num_workers=0,
        pin_memory=False,
    )
    return dataset, loader


## Probability Features

This cell converts model probabilities into the three router features: confidence, entropy, and margin between the top two probabilities.

In [5]:
def probability_features(probabilities):
    probabilities = np.asarray(probabilities)
    confidence = probabilities.max(axis=1)
    entropy = -(probabilities * np.log(probabilities + 1e-12)).sum(axis=1)
    top_two = np.partition(probabilities, -2, axis=1)[:, -2:]
    margin = top_two.max(axis=1) - top_two.min(axis=1)
    return np.column_stack([confidence, entropy, margin]).astype(np.float32)


def short_model_name(model_name):
    return model_name.replace("resnet", "RN").upper()


## Router Training Data

This cell builds router training data from the existing artifact NPZ files. It uses Model A probabilities as features and labels uncertain samples for Model B or Model C.

In [6]:
def load_cache(prefix, model_name):
    path = ARTIFACTS_DIR / f"{prefix}_{model_name}.npz"
    with np.load(path) as data:
        return {
            "probabilities": data["probabilities"],
            "predictions": data["predictions"],
            "labels": data["labels"],
        }


def cache_confidence(cache):
    return np.asarray(cache["probabilities"]).max(axis=1)


def build_router_training_data():
    feature_parts = []
    label_parts = []

    for prefix in ROUTER_TRAIN_PREFIXES:
        cache_a = load_cache(prefix, MODEL_A)
        cache_b = load_cache(prefix, MODEL_B)
        cache_c = load_cache(prefix, MODEL_C)

        features_a = probability_features(cache_a["probabilities"])
        model_a_uncertain = features_a[:, 0] < CONFIDENCE_THRESHOLD
        model_b_ok = cache_confidence(cache_b) >= CONFIDENCE_THRESHOLD
        model_c_ok = cache_confidence(cache_c) >= CONFIDENCE_THRESHOLD

        if ROUTER_REQUIRES_CORRECT:
            model_b_ok = model_b_ok & (cache_b["predictions"] == cache_a["labels"])
            model_c_ok = model_c_ok & (cache_c["predictions"] == cache_a["labels"])

        keep = model_a_uncertain & (model_b_ok | model_c_ok)
        route_labels = np.where(model_b_ok[keep], 0, 1).astype(np.int64)

        feature_parts.append(features_a[keep])
        label_parts.append(route_labels)

    router_features = np.concatenate(feature_parts)
    router_labels = np.concatenate(label_parts)
    if len(router_labels) == 0:
        raise ValueError("No router training rows found")
    return router_features, router_labels


router_train_features, router_train_labels = build_router_training_data()
print("Router training samples:", len(router_train_labels))
print(f"{MODEL_B} labels:", int(np.count_nonzero(router_train_labels == 0)))
print(f"{MODEL_C} labels:", int(np.count_nonzero(router_train_labels == 1)))


Router training samples: 2777
resnet34 labels: 2666
resnet50 labels: 111


## Random Forest Router

Run this cell to use a Random Forest router. To switch routers during Run All, comment out this cell and uncomment one of the next router cells.

In [7]:
# router_name = "rf"
# router = RandomForestClassifier(
#     n_estimators=100,
#     max_depth=6,
#     min_samples_leaf=20,
#     class_weight="balanced",
#     random_state=42,
# )
# router.fit(router_train_features, router_train_labels)
# print("Router:", router_name)


## Extra Trees Router

Uncomment and run this cell to use an Extra Trees router. Comment out the other router cells first.

In [8]:
# router_name = "extratree"
# router = ExtraTreesClassifier(
#     n_estimators=100,
#     max_depth=6,
#     min_samples_leaf=20,
#     class_weight="balanced",
#     random_state=42,
# )
# router.fit(router_train_features, router_train_labels)
# print("Router:", router_name)


## XGBoost Router

Uncomment and run this cell to use an XGBoost router. Comment out the other router cells first. Install xgboost before using it.

In [9]:
# Install xgboost before uncommenting this cell.
# from xgboost import XGBClassifier
#
# router_name = "xgboost"
# router = XGBClassifier(
#     n_estimators=100,
#     max_depth=4,
#     learning_rate=0.1,
#     eval_metric="logloss",
#     random_state=42,
# )
# router.fit(router_train_features, router_train_labels)
# print("Router:", router_name)


## SVM Router

Uncomment and run this cell to use an SVM router. Comment out the other router cells first.

In [10]:
router_name = "svm"
router = SVC(kernel="rbf", C=2.0, gamma="scale", class_weight="balanced")
router.fit(router_train_features, router_train_labels)
print("Router:", router_name)


Router: svm


## Test Dataset

This cell creates the local ImageNetV2 test loader and prints the selected worker devices.

In [11]:
if not torch.backends.mps.is_available():
    raise RuntimeError("MPS is required because Model B and Model C are configured for MPS")

real_test_dataset, real_test_loader = make_streaming_loader(TEST_VARIANT, MAX_SAMPLES)
real_sample_count = len(real_test_dataset)

print("Local samples:", real_sample_count)
print("Dataset:", VARIANT_ARCHIVES[TEST_VARIANT])
print("Worker limits:", WORKER_LIMITS)
print("Worker model devices:", MODEL_DEVICES)


Local samples: 10000
Dataset: /Users/abhinavgupta/dynamic-idk-cascades/ImageNet-V2 DataSet/imagenetv2-threshold0.7.tar.gz
Worker limits: {'cpu': 1, 'mps': 3}
Worker model devices: {'resnet18': 'mps', 'resnet34': 'mps', 'resnet50': 'mps'}


## Doubly Linked List

This cell defines the heavy-model waiting queue. Model B jobs stay at the head side; Model C jobs stay at the tail side; Model C can steal a Model B job when no Model C job is waiting.

In [12]:
class JobNode:
    def __init__(self, sample_index, images, assigned_model):
        self.sample_index = sample_index
        self.images = images
        self.assigned_model = assigned_model
        self.prev = None
        self.next = None


class DoublyLinkedList:
    def __init__(self, model_b, model_c):
        self.model_b = model_b
        self.model_c = model_c
        self.head = None
        self.tail = None
        self.last_model_b = None
        self.size = 0

    def _insert_empty(self, node):
        self.head = node
        self.tail = node
        self.size = 1
        if node.assigned_model == self.model_b:
            self.last_model_b = node

    def _insert_before(self, anchor, node):
        node.prev = anchor.prev
        node.next = anchor
        if anchor.prev is None:
            self.head = node
        else:
            anchor.prev.next = node
        anchor.prev = node
        self.size += 1

    def _insert_after(self, anchor, node):
        node.prev = anchor
        node.next = anchor.next
        if anchor.next is None:
            self.tail = node
        else:
            anchor.next.prev = node
        anchor.next = node
        self.size += 1

    def _remove(self, node):
        if node.prev is None:
            self.head = node.next
        else:
            node.prev.next = node.next

        if node.next is None:
            self.tail = node.prev
        else:
            node.next.prev = node.prev

        if node is self.last_model_b:
            self.last_model_b = node.prev if node.prev and node.prev.assigned_model == self.model_b else None

        node.prev = None
        node.next = None
        self.size -= 1

        if self.size == 0:
            self.head = None
            self.tail = None
            self.last_model_b = None

        return node

    def insert_middle(self, sample_index, images, assigned_model):
        node = JobNode(sample_index, images, assigned_model)
        if self.size == 0:
            self._insert_empty(node)
            return node

        if assigned_model == self.model_b:
            if self.last_model_b is None:
                self._insert_before(self.head, node)
            else:
                self._insert_after(self.last_model_b, node)
            self.last_model_b = node
        elif assigned_model == self.model_c:
            if self.last_model_b is None:
                self._insert_before(self.head, node)
            else:
                self._insert_after(self.last_model_b, node)
        else:
            raise ValueError(f"Unknown heavy model: {assigned_model}")

        return node

    def pop_for_model_b(self):
        if self.head is None or self.head.assigned_model != self.model_b:
            return None
        return self._remove(self.head)

    def pop_for_model_c_or_steal_model_b(self):
        if self.tail is not None and self.tail.assigned_model == self.model_c:
            return self._remove(self.tail), False

        node = self.pop_for_model_b()
        if node is None:
            return None, False
        return node, True


## Router Function

This cell converts Model A probabilities into router features and returns Model B or Model C.

In [13]:
def route_from_model_a(probabilities):
    route_label = int(router.predict(probability_features(probabilities[None, :]))[0])
    return MODEL_B if route_label == 0 else MODEL_C


## Real-Time Run

This cell starts one CPU worker for Model A and two MPS workers for Model B and Model C, routes uncertain Model A samples, and records all metrics.

In [14]:
def run_real_time_test():
    mp.set_start_method("spawn", force=True)
    ctx = mp.get_context("spawn")
    job_queues = {model_name: ctx.SimpleQueue() for model_name in MODELS}
    result_queue = ctx.SimpleQueue()
    processes = [
        ctx.Process(
            target=model_worker,
            args=(model_name, MODEL_DEVICES[model_name], job_queues[model_name], result_queue),
        )
        for model_name in MODELS
    ]

    for process in processes:
        process.start()

    try:
        total_samples = len(real_test_dataset)
        labels = np.full(total_samples, -1, dtype=np.int64)
        final_predictions = np.full(total_samples, -1, dtype=np.int64)
        chosen_models = np.full(total_samples, "", dtype="<U32")
        latencies_ms = np.full(total_samples, np.nan, dtype=np.float64)

        execution_count_by_model = Counter()
        execution_time_ms_by_model = Counter()
        heavy_route_count_by_model = Counter()
        stolen_job_count_by_model = Counter()
        in_flight_by_model = Counter()

        sample_states = {}
        doubly_linked_list = DoublyLinkedList(MODEL_B, MODEL_C)
        loader_iterator = iter(real_test_loader)
        next_sample_index = 0
        completed_sample_count = 0
        heavy_queue_max_size = 0
        router_call_count = 0
        router_time_ms_total = 0.0
        run_start = time.perf_counter()

        def heavy_backlog_size():
            return doubly_linked_list.size + sum(in_flight_by_model[model_name] for model_name in MPS_MODELS)

        def finish_sample(sample_index, model_name, prediction):
            final_predictions[sample_index] = prediction
            chosen_models[sample_index] = model_name
            latencies_ms[sample_index] = (time.perf_counter() - sample_states[sample_index]["start_time"]) * 1000.0
            sample_states.pop(sample_index)

        def dispatch_mps_jobs(max_size):
            for model_name in MPS_MODELS:
                while in_flight_by_model[model_name] < MAX_IN_FLIGHT_PER_MPS_MODEL:
                    if model_name == MODEL_B:
                        node = doubly_linked_list.pop_for_model_b()
                        stole_model_b_job = False
                    else:
                        node, stole_model_b_job = doubly_linked_list.pop_for_model_c_or_steal_model_b()

                    if node is None:
                        break

                    job_queues[model_name].put((node.sample_index, node.images))
                    in_flight_by_model[model_name] += 1
                    execution_count_by_model[model_name] += 1

                    if stole_model_b_job:
                        stolen_job_count_by_model[MODEL_C] += 1

                    max_size = max(max_size, heavy_backlog_size())
            return max_size

        while completed_sample_count < total_samples:
            while in_flight_by_model[MODEL_A] < WORKER_LIMITS["cpu"] and next_sample_index < total_samples:
                images, batch_labels = next(loader_iterator)
                sample_index = next_sample_index
                next_sample_index += 1
                labels[sample_index] = int(batch_labels.item())
                sample_states[sample_index] = {"images": images, "start_time": time.perf_counter()}
                job_queues[MODEL_A].put((sample_index, images))
                in_flight_by_model[MODEL_A] += 1
                execution_count_by_model[MODEL_A] += 1

            message = result_queue.get()
            if message[0] == "error":
                _, model_name, error = message
                raise RuntimeError(f"{model_name} worker failed: {error}")

            sample_index, model_name, probabilities, prediction, confidence, elapsed_ms = message
            in_flight_by_model[model_name] -= 1
            execution_time_ms_by_model[model_name] += elapsed_ms

            if model_name == MODEL_A and confidence < CONFIDENCE_THRESHOLD:
                router_start = time.perf_counter()
                assigned_model = route_from_model_a(probabilities)
                router_time_ms_total += (time.perf_counter() - router_start) * 1000.0
                router_call_count += 1
                heavy_route_count_by_model[assigned_model] += 1
                doubly_linked_list.insert_middle(sample_index, sample_states[sample_index]["images"], assigned_model)
                sample_states[sample_index]["images"] = None
                heavy_queue_max_size = max(heavy_queue_max_size, heavy_backlog_size())
            else:
                finish_sample(sample_index, model_name, prediction)
                completed_sample_count += 1

            heavy_queue_max_size = dispatch_mps_jobs(heavy_queue_max_size)

        total_wall_time_seconds = time.perf_counter() - run_start
        correct_predictions = int(np.count_nonzero(final_predictions == labels))
        total_execution_time_ms_by_model = {model_name: float(execution_time_ms_by_model[model_name]) for model_name in MODELS}
        mean_execution_time_ms_by_model = {
            model_name: float(execution_time_ms_by_model[model_name] / execution_count_by_model[model_name])
            if execution_count_by_model[model_name]
            else 0.0
            for model_name in MODELS
        }

        results = {
            "total_samples": int(total_samples),
            "accuracy": float(correct_predictions / total_samples),
            "correct_predictions": int(correct_predictions),
            "total_wall_time_seconds": float(total_wall_time_seconds),
            "throughput_fps": float(total_samples / total_wall_time_seconds),
            "mean_latency_ms": float(latencies_ms.mean()),
            "worker_limits": dict(WORKER_LIMITS),
            "device_by_model": {model_name: MODEL_DEVICES[model_name] for model_name in MODELS},
            "router_name": router_name,
            "router_call_count": int(router_call_count),
            "total_router_time_ms": float(router_time_ms_total),
            "mean_router_time_ms": float(router_time_ms_total / router_call_count) if router_call_count else 0.0,
            "stolen_job_count_by_model": {model_name: int(stolen_job_count_by_model[model_name]) for model_name in MODELS},
            "final_prediction_count_by_model": {model_name: int(np.count_nonzero(chosen_models == model_name)) for model_name in MODELS},
            "execution_count_by_model": {model_name: int(execution_count_by_model[model_name]) for model_name in MODELS},
            "mean_execution_time_ms_by_model": mean_execution_time_ms_by_model,
            "idle_time_ms_by_model": {
                model_name: max(0.0, total_wall_time_seconds * 1000.0 - total_execution_time_ms_by_model[model_name])
                for model_name in MODELS
            },
            "heavy_route_count_by_model": {model_name: int(heavy_route_count_by_model[model_name]) for model_name in MPS_MODELS},
            "heavy_queue_max_size": int(heavy_queue_max_size),
        }
        return results, final_predictions, chosen_models

    finally:
        for queue in job_queues.values():
            queue.put(None)
        for process in processes:
            process.join()


real_system_results, real_final_predictions, real_chosen_models = run_real_time_test()


## Print and Save Metrics

This cell prints the requested real-time metrics and saves the JSON results plus NPZ predictions when SAVE_RESULTS is True.

In [15]:
print("Real-Time CPU/MPS Multiprocessing Test")
print("Confidence Threshold Constant: ", CONFIDENCE_THRESHOLD)
print("Max in flight: ", MAX_IN_FLIGHT_PER_MPS_MODEL)
print("Worker limits:", real_system_results["worker_limits"])
print("Device by model:", real_system_results["device_by_model"])
print("Total samples:", real_system_results["total_samples"])
print("Accuracy:", round(real_system_results["accuracy"], 4))
print("Correct predictions:", real_system_results["correct_predictions"])
print("Total wall time (seconds):", round(real_system_results["total_wall_time_seconds"], 3))
print("Throughput (FPS):", round(real_system_results["throughput_fps"], 3))
print("Mean latency (ms):", round(real_system_results["mean_latency_ms"], 3))
print()
print(f"Classifier {real_system_results['router_name']} router:")
print(f"  Router calls: {real_system_results['router_call_count']}")
print(f"  Total router time (ms): {real_system_results['total_router_time_ms']:.3f}")
print(f"  Mean router time (ms): {real_system_results['mean_router_time_ms']:.6f}")
print()
print(
    f"# of jobs stolen by {short_model_name(MODEL_C)} originally assigned to {short_model_name(MODEL_B)}:",
    real_system_results["stolen_job_count_by_model"][MODEL_C],
)
print()
print("Final prediction count by model:")
for model_name in MODELS:
    print(f"  {model_name}: {real_system_results['final_prediction_count_by_model'][model_name]}")
print("Execution count by model:")
for model_name in MODELS:
    print(f"  {model_name}: {real_system_results['execution_count_by_model'][model_name]}")
print("Mean execution time by model (ms):")
for model_name in MODELS:
    print(f"  {model_name}: {real_system_results['mean_execution_time_ms_by_model'][model_name]:.3f}")
print("Idle time by model (seconds):")
for model_name in MODELS:
    idle_seconds = real_system_results["idle_time_ms_by_model"][model_name] / 1000.0
    print(f"  {model_name}: {idle_seconds:.3f}")
print("Heavy route count by MPS model:")
for model_name in MPS_MODELS:
    print(f"  {model_name}: {real_system_results['heavy_route_count_by_model'][model_name]}")
print("Heavy queue max size:", real_system_results["heavy_queue_max_size"])

if SAVE_RESULTS:
    RESULTS_PATH.write_text(json.dumps(real_system_results, indent=2) + "\n", encoding="utf-8")
    np.savez_compressed(
        PREDICTIONS_PATH,
        predictions=real_final_predictions,
        chosen_models=real_chosen_models,
    )
    print("Saved:", RESULTS_PATH.name)
    print("Saved:", PREDICTIONS_PATH.name)
else:
    print("SAVE_RESULTS is False; no files were written.")


Real-Time CPU/MPS Multiprocessing Test
Confidence Threshold Constant:  0.7
Max in flight:  1000
Worker limits: {'cpu': 1, 'mps': 3}
Device by model: {'resnet18': 'mps', 'resnet34': 'mps', 'resnet50': 'mps'}
Total samples: 10000
Accuracy: 0.7318
Correct predictions: 7318
Total wall time (seconds): 170.924
Throughput (FPS): 58.506
Mean latency (ms): 24.956

Classifier svm router:
  Router calls: 4445
  Total router time (ms): 1637.976
  Mean router time (ms): 0.368498

# of jobs stolen by RN50 originally assigned to RN34: 0

Final prediction count by model:
  resnet18: 5555
  resnet34: 2555
  resnet50: 1890
Execution count by model:
  resnet18: 10000
  resnet34: 2555
  resnet50: 1890
Mean execution time by model (ms):
  resnet18: 12.624
  resnet34: 18.784
  resnet50: 28.126
Idle time by model (seconds):
  resnet18: 44.683
  resnet34: 122.931
  resnet50: 117.766
Heavy route count by MPS model:
  resnet34: 2555
  resnet50: 1890
Heavy queue max size: 3
Saved: real_cpu_mps_worker_results.jso